# Scenarios

Demonstrate scenarios for satellite geolocation.

In [ ]:
from datetime import timedelta

from matplotlib import pyplot as plt
from skyfield.api import load
from skyfield.iokit import parse_tle_file

## Two Satellites

In [ ]:
ts = load.timescale()

with load.open("g16.tle", 'rb') as f:
    satellites = list(parse_tle_file(f, ts))

es_g16 = satellites[0]

with load.open("g19.tle", 'rb') as f:
    satellites = list(parse_tle_file(f, ts))

es_g19 = satellites[0]

es_g16, es_g19

In [ ]:
epoch_dt = es_g16.epoch.utc_datetime()

# Configuration for time increments
increment_minutes = 15
half_day = timedelta(hours=12)

# Generate list of datetimes from -0.5 days to +0.5 days from epoch_dt
start_dt = epoch_dt - half_day
end_dt = epoch_dt + half_day

dt_observables = []
observe_dt = start_dt
while observe_dt <= end_dt:
    dt_observables.append(observe_dt)
    observe_dt += timedelta(minutes=increment_minutes)

print(f"Generated {len(dt_observables)} datetime objects.")

Two antennas will be pointed to the respective satellites. We will again use a known emitter, the [Montana PBS](https://en.wikipedia.org/wiki/Montana_PBS).

In [ ]:
from sat_geo_solver.scenarios import TwoSat

cos_ll = [+38.4, -104.82]
cos_ll_2 = [+38.4001, -104.8201]
montana_pbs_ll = [45.6655208,-111.0514232, 271]

two_sat = TwoSat(
    primary_satellite=es_g16,
    secondary_satellite=es_g19,
    at=dt_observables,
    primary_receiver=cos_ll,
    secondary_receiver=cos_ll_2,
)

### Differential Time Offset

Demonstrate differential time offset calculation between two satellites over time.

In [ ]:
primary_signal_path = two_sat.primary.light_seconds(montana_pbs_ll) + two_sat.primary.light_seconds(cos_ll)
secondary_signal_path = two_sat.secondary.light_seconds(montana_pbs_ll) + two_sat.secondary.light_seconds(cos_ll_2)

In [ ]:
plt.plot(primary_signal_path, label="Primary")
plt.plot(secondary_signal_path, label="Secondary")
plt.ylabel("Light Time (s)")
plt.legend()
plt.grid();

In [ ]:
dto = two_sat.dto(from_pos=montana_pbs_ll)

In [ ]:
plt.plot(dto)
plt.ylabel("Differential Time Offset (s)")
plt.grid();

## Different Frequency Offset

Demonstrate differential frequency offset calculation between two satellites over time.

In [ ]:
translation_frequency = 2.3e9
pbs_downlink = 12.14e9
pbs_uplink = pbs_downlink + translation_frequency

primary_signal_doppler = two_sat.primary.downlink_received_frequency(from_pos=montana_pbs_ll, to_pos=cos_ll, uplink=pbs_uplink, translation_frequency=translation_frequency)
secondary_signal_doppler = two_sat.secondary.downlink_received_frequency(from_pos=montana_pbs_ll, to_pos=cos_ll_2, uplink=pbs_uplink, translation_frequency=translation_frequency)

In [ ]:
plt.plot(primary_signal_doppler, label="Primary")
plt.plot(secondary_signal_doppler, label="Secondary")
plt.ylabel("Doppler Shift (Hz)")
plt.legend()
plt.grid();

In [ ]:
dfo = two_sat.dfo(from_pos=montana_pbs_ll, uplink=pbs_uplink, translation_frequency=translation_frequency)

In [ ]:
plt.plot(dfo)
plt.ylabel("Different Frequency Offset (Hz)")
plt.grid();